In [0]:
from pyspark.sql.functions import (
    col,
    explode,
    lit,
    current_timestamp,
    input_file_name
)

RAW_BASE_PATH = "/Volumes/workspace/fhir/raw_data"

print("Bronze notebook started")
print("Raw path:", RAW_BASE_PATH)

In [0]:
RESOURCE_TYPES = [
    "Patient",
    "Encounter",
    "Observation",
    "Condition"
]

print(RESOURCE_TYPES)

In [0]:
extraction_date = "2026-09-10"

patient_raw_path = (
    f"{RAW_BASE_PATH}/Patient/"
    f"extraction_date={extraction_date}"
)

display(
    dbutils.fs.ls(patient_raw_path)
)

In [0]:
patient_raw_df = (
    spark.read
    .option("multiLine", "true")
    .json(patient_raw_path)
)

display(patient_raw_df)

In [0]:
patient_raw_df.printSchema()

In [0]:
patient_entries_df = (
    patient_raw_df
    .select(
        explode(col("entry")).alias("entry"),
        col("_metadata.file_path").alias("source_file")
    )
)

display(patient_entries_df)

In [0]:
patient_bronze_df = (
    patient_entries_df
    .select(
        col("entry.fullUrl").alias("full_url"),
        col("entry.resource").alias("resource"),
        col("source_file")
    )
)

display(patient_bronze_df)

In [0]:
patient_bronze_df = (
    patient_bronze_df
    .withColumn(
        "resource_type",
        col("resource.resourceType")
    )
    .withColumn(
        "resource_id",
        col("resource.id")
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

display(patient_bronze_df)

In [0]:
from pyspark.sql.functions import to_json

patient_bronze_df = (
    patient_bronze_df
    .withColumn(
        "raw_json",
        to_json(col("resource"))
    )
    .drop("resource")
)

display(patient_bronze_df)

In [0]:
print("Patient Bronze records:", patient_bronze_df.count())

patient_bronze_df.select(
    "resource_id",
    "resource_type",
    "source_file",
    "ingestion_timestamp"
).show(5, truncate=False)

In [0]:
(
    patient_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.fhir.bronze_patient")
)

print("bronze_patient created successfully")

In [0]:
display(
    spark.table("workspace.fhir.bronze_patient")
)

In [0]:
print(
    "Bronze Patient count:",
    spark.table("workspace.fhir.bronze_patient").count()
)

In [0]:
def create_bronze_table(resource_type, extraction_date):

    raw_path = (
        f"{RAW_BASE_PATH}/"
        f"{resource_type}/"
        f"extraction_date={extraction_date}"
    )

    print("====================================")
    print("Resource:", resource_type)
    print("Reading:", raw_path)
    print("====================================")

    # 1. Read Raw JSON Bundle
    raw_df = (
        spark.read
        .option("multiLine", "true")
        .json(raw_path)
    )

    # 2. Explode FHIR Bundle entries
    entries_df = (
        raw_df
        .select(
            explode(col("entry")).alias("entry"),
            col("_metadata.file_path").alias("source_file")
        )
    )

    # 3. Extract actual FHIR resource
    bronze_df = (
        entries_df
        .select(
            col("entry.fullUrl").alias("full_url"),
            col("entry.resource").alias("resource"),
            col("source_file")
        )
    )

    # 4. Add metadata
    bronze_df = (
        bronze_df
        .withColumn(
            "resource_type",
            col("resource.resourceType")
        )
        .withColumn(
            "resource_id",
            col("resource.id")
        )
        .withColumn(
            "ingestion_timestamp",
            current_timestamp()
        )
    )

    # 5. Preserve complete FHIR resource as JSON
    bronze_df = (
        bronze_df
        .withColumn(
            "raw_json",
            to_json(col("resource"))
        )
        .drop("resource")
    )

    # 6. Target table name
    table_name = (
        f"workspace.fhir.bronze_"
        f"{resource_type.lower()}"
    )

    # 7. Write Delta table
    (
        bronze_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

    # 8. Count records
    record_count = bronze_df.count()

    print(f"Created table: {table_name}")
    print(f"Records: {record_count}")

    return record_count

In [0]:
from datetime import datetime, timezone

extraction_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

print("Extraction date:", extraction_date)

In [0]:
RESOURCE_TYPES = [
    "Patient",
    "Encounter",
    "Observation",
    "Condition"
]

bronze_counts = {}

for resource_type in RESOURCE_TYPES:

    count = create_bronze_table(
        resource_type=resource_type,
        extraction_date=extraction_date
    )

    bronze_counts[resource_type] = count

In [0]:
for resource_type, count in bronze_counts.items():

    print(
        f"{resource_type}: {count} records"
    )

In [0]:
for resource_type in RESOURCE_TYPES:

    table_name = (
        f"workspace.fhir.bronze_"
        f"{resource_type.lower()}"
    )

    print(
        f"{table_name}: "
        f"{spark.table(table_name).count()} records"
    )

In [0]:
spark.sql("""
SHOW TABLES IN workspace.fhir
""").show(truncate=False)

In [0]:
display(
    spark.table("workspace.fhir.bronze_patient")
    .select(
        "resource_id",
        "resource_type",
        "raw_json",
        "ingestion_timestamp"
    )
    .limit(3)
)

In [0]:
display(
    spark.table("workspace.fhir.bronze_encounter")
    .select(
        "resource_id",
        "resource_type",
        "raw_json",
        "ingestion_timestamp"
    )
    .limit(3)
)

In [0]:
display(
    spark.table("workspace.fhir.bronze_observation")
    .select(
        "resource_id",
        "resource_type",
        "raw_json",
        "ingestion_timestamp"
    )
    .limit(3)
)

In [0]:
display(
    spark.table("workspace.fhir.bronze_condition")
    .select(
        "resource_id",
        "resource_type",
        "raw_json",
        "ingestion_timestamp"
    )
    .limit(3)
)

In [0]:
for resource_type in RESOURCE_TYPES:

    table_name = (
        f"workspace.fhir.bronze_"
        f"{resource_type.lower()}"
    )

    df = spark.table(table_name)

    print("================================")
    print("Table:", table_name)
    print("Count:", df.count())
    print("Columns:", df.columns)